# EARC Pipeline — Modules 1-3 Test (Layers 1-9)

This notebook runs the full **Retrieval (1-3) → Scoring (4-6) → Selection (7-9)** flow on Colab using the corpus artifacts stored in Google Drive.

**What you get per query:**
- Module 1: query analysis + retrieved/segmented sentences
- Module 2: query-first embeddings + multi-signal scores + redundancy removal
- Module 3: reasoning-graph bridges + token-budget selection + diversity guard

> Run the cells top-to-bottom. Steps 1-5 are one-time setup; Steps 6+ are the actual tests.

## Step 1 — Mount Google Drive
Your FAISS index, BM25 index, chunk shards and metadata shards live here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Get the code

Pick **one** option below.

- **Option A (GitHub):** clone the repo. Make sure you have **pushed** the latest Module 2/3 changes first, otherwise you'll get the old stubs.
- **Option B (Drive):** if you keep the project folder in Drive, set `REPO_DIR` to that path and skip the clone.

In [ ]:
import os, sys

# ---- Option A: clone from GitHub (uncomment) ----
# !rm -rf /content/earc-pipeline
# !git clone https://github.com/anaghadk/EARC_Pipeline.git /content/earc-pipeline
# REPO_DIR = '/content/earc-pipeline'

# ---- Option B: use a copy already in Drive (edit path) ----
REPO_DIR = '/content/drive/MyDrive/RAG_Project/earc-pipeline'

assert os.path.isdir(REPO_DIR), f'REPO_DIR not found: {REPO_DIR}'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Using repo at:', REPO_DIR)
print(os.listdir(REPO_DIR))

## Step 3 — Install dependencies
Installs the pinned requirements and the spaCy English model. Safe to re-run.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print('Dependencies installed.')

## Step 4 — Point to your corpus artifacts

These are the same files Module 1 already uses. Edit the paths if your Drive layout differs.
Expected layout:
```
RAG_Project/
  faiss.index
  bm25.pkl
  chunks/      (chunks_*.pkl)
  metadata/    (metadata_*.pkl)
```

In [ ]:
from pathlib import Path

DRIVE_BASE   = Path('/content/drive/MyDrive/RAG_Project')
FAISS_PATH   = DRIVE_BASE / 'faiss.index'
BM25_PATH    = DRIVE_BASE / 'bm25.pkl'
CHUNKS_DIR   = DRIVE_BASE / 'chunks'
METADATA_DIR = DRIVE_BASE / 'metadata'

for p in [FAISS_PATH, BM25_PATH, CHUNKS_DIR, METADATA_DIR]:
    print(('OK   ' if p.exists() else 'MISSING ') + str(p))

## Step 5 — Initialise the full pipeline (Modules 1-3)

`EARCPipeline` loads all artifacts once and chains:
Retrieval (1-3) → Scoring (4-6) → Selection (7-9).

First run takes a minute (downloads embedding model + loads FAISS/BM25).

The **version guard** at the end fails loudly if Colab is still running the OLD code
(before Module 2/3 were wired in). If it trips: re-sync the repo and restart the runtime.

In [ ]:
from pipeline import EARCPipeline

pipe = EARCPipeline(
    faiss_path=FAISS_PATH,
    bm25_path=BM25_PATH,
    chunks_dir=CHUNKS_DIR,
    metadata_dir=METADATA_DIR,
)

# Version guard: confirm Colab is running the UPDATED code (Modules 2 & 3).
# If either assert fails, Colab loaded an OLD copy of the repo.
# Fix: re-sync the code (git push + re-clone, or replace the Drive copy),
# then Runtime > Restart runtime and re-run from Step 5.
assert hasattr(pipe, 'scoring_pipeline'), (
    'OLD CODE: EARCPipeline has no scoring_pipeline. Re-sync repo + restart runtime.'
)
assert hasattr(pipe, 'selection_pipeline'), (
    'OLD CODE: EARCPipeline has no selection_pipeline. Re-sync repo + restart runtime.'
)
print('\nEARCPipeline ready — Modules 1-3 wired in. Code version OK.')

## Step 6 — Pretty-printer for all 3 modules
A small helper that shows each module's contribution for a single query.

In [ ]:
def show_result(result, top_k_scored=8, top_k_selected=12):
    qi = result['query_info']
    print('=' * 80)
    print('QUERY:', result['query'])
    print('=' * 80)

    # ---- Module 1: Query analysis + retrieval ----
    print('\n[MODULE 1] Query Analysis & Retrieval')
    print('  query_type   :', qi['query_type'])
    print('  has_negation :', qi['has_negation'])
    print('  keywords     :', qi['keywords'])
    print('  entities     :', qi['entities'])
    print('  scored sents :', len(result['sentences']))

    # ---- Module 2: Scoring ----
    print('\n[MODULE 2] Top scored sentences (after embedding + dedup)')
    top = sorted(result['sentences'], key=lambda s: s.final_score, reverse=True)[:top_k_scored]
    for i, s in enumerate(top, 1):
        print(f'  {i:2d}. score={s.final_score:.4f}  '
              f'(sim={s.semantic_score:.2f} ev={s.evidence_score:.2f} '
              f'evd={s.evidentiality_score:.2f} dens={s.claim_density_score:.2f})  '
              f'[{s.dataset}/{s.doc_id}]')
        print('      ', s.text[:160])

    # ---- Module 3: Selection ----
    if 'selected_sentences' not in result:
        print('\n[MODULE 3] MISSING. You are running OLD code (no Module 2/3 output).')
        print('  Fix: re-sync the repo to Colab, then Runtime > Restart runtime.')
        return
    sel = result['selected_sentences']
    print(f'\n[MODULE 3] Selected evidence: {len(sel)} sentences')
    for i, s in enumerate(sel[:top_k_selected], 1):
        print(f'  {i:2d}. score={float(s["score"]):.4f}  bridge={s["is_bridge"]}  '
              f'[{s.get("dataset", "?")}/{s["doc_id"]}]')
        print('      ', s['text'][:160])

    # ---- Selection stats ----
    stats = result['selection_stats']
    print('\n[MODULE 3] Stats')
    print('  reasoning:', stats.get('reasoning'))
    print('  budget   :', stats.get('budget'))
    print('  diversity:', {k: stats.get('diversity', {}).get(k) for k in
          ('coverage_ratio', 'keyword_coverage_ratio', 'sentence_swaps', 'coverage_improved')})
    print('=' * 80)

## Step 7 — Run a single query
Change the text and re-run to test interactively.

In [ ]:
result = pipe.run('Who invented the telephone?')
show_result(result)

## Step 8 — Batch test across query types
Covers factoid, multi-hop and negation queries so you can confirm classification + selection behave per type.

In [ ]:
test_queries = [
    'Who invented the telephone?',
    'What did Marie Curie and Albert Einstein both contribute to physics?',
    'What countries are not members of NATO?',
    'How does photosynthesis work?',
]

for q in test_queries:
    res = pipe.run(q)
    show_result(res, top_k_scored=4, top_k_selected=6)
    print('\n')

## Step 9 — (Optional) Inspect each module in isolation
Useful for debugging: runs the modules step-by-step and shows the intermediate hand-off so you can confirm each boundary is healthy.

In [ ]:
query = 'What did Marie Curie and Albert Einstein both contribute to physics?'

# --- Module 1 ---
sentences, query_info = pipe.retrieval_layer.retrieve(query)
print('M1 ->', len(sentences), 'sentences | type =', query_info['query_type'])
assert sentences and sentences[0].embedding is None, 'M1 should leave embedding=None'

# --- Module 2 ---
scored = pipe.scoring_pipeline.run(query_info, sentences)
print('M2 ->', len(scored), 'scored+deduped | embedding set =', scored[0].embedding is not None)
records = pipe.scoring_pipeline.to_selection_records(scored)
print('M2 -> adapter records:', len(records), '| keys =', sorted(records[0].keys())[:6], '...')

# --- Module 3 ---
selection = pipe.selection_pipeline.run(query_info, records)
print('M3 -> selected =', len(selection['selected_sentences']),
      '| candidates =', len(selection['candidate_sentences']))
print('M3 -> stats keys:', list(selection['stats'].keys()))

## Step 10 — (Optional) Export selected evidence for teammates
Saves Module 1-3 output to Drive as a pickle, so Module 4 (generation) work can start without re-running retrieval.

In [ ]:
import pickle, re
from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/RAG_Project/earc_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export_for_module4(query):
    result = pipe.run(query)
    slug = re.sub(r'[^a-z0-9]+', '_', query.lower())[:40]
    out_path = OUTPUT_DIR / f'm123_{slug}.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump({
            'query': result['query'],
            'query_info': result['query_info'],
            'selected_sentences': result['selected_sentences'],
            'candidate_sentences': result['candidate_sentences'],
            'selection_stats': result['selection_stats'],
        }, f)
    print(f'Saved {len(result["selected_sentences"])} selected sentences -> {out_path.name}')
    return out_path

export_for_module4('Who invented the telephone?')